# 店舗売上需要予測: 実験 #008 再帰的マルチステップ予測（直近ラグ完全解禁）

- **コンペ名:** [Store Sales - Time Series Forecasting](https://www.kaggle.com/competitions/store-sales-time-series-forecasting)
- **目的:** Direct予測の制約（Lag 16以上のみ）を打ち破り、**直近ラグ（Lag 1, 2, 3, 7, 14）と移動平均（7, 14, 28日）を完全解禁**。1日ずつ推論して当日の予測値を翌日のラグにフィードバックする **再帰的自己回帰予測（Recursive Roll-forward）** を実装する。
- **モデル:** 33商品ファミリー別 3大GBDTアンサンブル (LightGBM 50% + CatBoost 30% + XGBoost 20%)
- **評価指標:** **RMSLE (Root Mean Squared Logarithmic Error)**
- **前回到達:** 実験 #006: **LB 0.41064 (155位 / 上位28%)**
- **本実験達成:** **LB `0.39084` (82位 / 73人抜き！念願の100位切り・2桁順位達成！🎉)**

---

## 1. 再帰的予測（Recursive Forecasting）とは？

1. **従来の制約 (Direct Forecasting):**
   - 16日間のテスト期間を一括予測するため、全日程で利用可能な「16日前（Lag 16）以前」の実績しか使えなかった。
   - Day 1（8/16）〜Day 7（8/22）にとって、直近1週間の需要トレンドを完全に捨てていた。
2. **再帰的予測のブレイクスルー:**
   - Day 1 は前日（8/15＝Lag 1）や前週（8/9＝Lag 7）の実績をフル活用して予測。
   - 予測した売上をデータフレームに書き戻し、Day 2 の Lag 1 として使う。
   - これを 16 日間繰り返すことで、テスト前半の予測精度が飛躍的に高まる！

## 2. ライブラリ読み込み & データ準備

In [ ]:
import os
import sys
import glob
import warnings
from datetime import timedelta
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error
import lightgbm as lgb
import catboost as cb
import xgboost as xgb

warnings.filterwarnings('ignore', category=UserWarning)

# データディレクトリ自動検出
def find_data_dir():
    candidates = [
        '/kaggle/input/store-sales-time-series-forecasting',
        '/kaggle/input/competitions/store-sales-time-series-forecasting',
        '003_store-sales/data',
        'data',
        '../data',
    ]
    for c in candidates:
        if os.path.exists(os.path.join(c, 'train.csv')):
            return c
    matches = glob.glob('/kaggle/input/**/train.csv', recursive=True)
    if matches:
        return os.path.dirname(matches[0])
    matches_local = glob.glob('**/train.csv', recursive=True)
    if matches_local:
        return os.path.dirname(matches_local[0])
    raise FileNotFoundError('train.csv が見つかりませんでした。')

data_dir = find_data_dir()
print(f'データ読み込み元: {data_dir}')

train_df = pd.read_csv(os.path.join(data_dir, 'train.csv'))
test_df = pd.read_csv(os.path.join(data_dir, 'test.csv'))
train_df['date'] = pd.to_datetime(train_df['date'])
test_df['date'] = pd.to_datetime(test_df['date'])


## 3. 特徴量エンジニアリング（直近ラグ & 特売先行・後行）

In [ ]:
# ゼロ売上系列の抽出
recent_train = train_df[train_df['date'] >= '2017-07-25']
recent_sum = recent_train.groupby(['store_nbr', 'family'])['sales'].sum().reset_index()
zero_pairs = set(zip(recent_sum[recent_sum['sales'] == 0]['store_nbr'], recent_sum[recent_sum['sales'] == 0]['family']))

pre_val = train_df[(train_df['date'] >= '2017-07-10') & (train_df['date'] < '2017-07-31')]
zero_pre_val = pre_val.groupby(['store_nbr', 'family'])['sales'].sum().reset_index()
val_zero_pairs = set(zip(zero_pre_val[zero_pre_val['sales'] == 0]['store_nbr'], zero_pre_val[zero_pre_val['sales'] == 0]['family']))

# 全体結合
test_df['sales'] = np.nan
all_df = pd.concat([train_df, test_df], ignore_index=True)
all_df['log_sales'] = np.log1p(all_df['sales'])

# カレンダー特徴量
all_df['year'] = all_df['date'].dt.year
all_df['month'] = all_df['date'].dt.month
all_df['day'] = all_df['date'].dt.day
all_df['dayofweek'] = all_df['date'].dt.dayofweek
all_df['is_weekend'] = all_df['dayofweek'].isin([5, 6]).astype(int)
all_df['dayofyear'] = all_df['date'].dt.dayofyear
all_df['weekofyear'] = all_df['date'].dt.isocalendar().week.astype(int)
all_df['is_month_start'] = all_df['date'].dt.is_month_start.astype(int)
all_df['is_month_end'] = all_df['date'].dt.is_month_end.astype(int)
all_df['is_wage_day'] = ((all_df['day'] == 15) | (all_df['is_month_end'] == 1)).astype(int)
all_df['is_wage_day_after'] = ((all_df['day'] == 16) | (all_df['is_month_start'] == 1)).astype(int)

# 特売特徴量
all_df = all_df.sort_values(['store_nbr', 'family', 'date']).reset_index(drop=True)
group_promo = all_df.groupby(['store_nbr', 'family'])['onpromotion']
all_df['promo_lag_1'] = group_promo.shift(1).fillna(0)
all_df['promo_lag_2'] = group_promo.shift(2).fillna(0)
all_df['promo_lag_7'] = group_promo.shift(7).fillna(0)
all_df['promo_lead_1'] = group_promo.shift(-1).bfill().fillna(0)
all_df['promo_lead_2'] = group_promo.shift(-2).bfill().fillna(0)
all_df['promo_diff_1'] = all_df['onpromotion'] - all_df['promo_lag_1']
all_df['promo_rolling_mean_7'] = group_promo.transform(lambda x: x.rolling(7, min_periods=1).mean())

# 過去売上ラグ（学習用）
group_sales = all_df.groupby(['store_nbr', 'family'])['log_sales']
for lag in [1, 2, 3, 7, 14, 21, 28, 364]:
    all_df[f'sales_lag_{lag}'] = group_sales.shift(lag)
all_df['sales_lag_364'] = all_df['sales_lag_364'].fillna(all_df['sales_lag_28'])

lag1 = group_sales.shift(1)
all_df['sales_rolling_mean_7'] = lag1.groupby([all_df['store_nbr'], all_df['family']]).transform(lambda x: x.rolling(7, min_periods=1).mean())
all_df['sales_rolling_mean_14'] = lag1.groupby([all_df['store_nbr'], all_df['family']]).transform(lambda x: x.rolling(14, min_periods=1).mean())
all_df['sales_rolling_mean_28'] = lag1.groupby([all_df['store_nbr'], all_df['family']]).transform(lambda x: x.rolling(28, min_periods=1).mean())

features = [
    'store_nbr', 'onpromotion', 'year', 'month', 'day', 'dayofweek', 'is_weekend',
    'dayofyear', 'weekofyear', 'is_month_start', 'is_month_end', 'is_wage_day', 'is_wage_day_after',
    'sales_lag_1', 'sales_lag_2', 'sales_lag_3', 'sales_lag_7', 'sales_lag_14',
    'sales_lag_21', 'sales_lag_28', 'sales_lag_364',
    'sales_rolling_mean_7', 'sales_rolling_mean_14', 'sales_rolling_mean_28',
    'promo_lag_1', 'promo_lag_2', 'promo_lag_7',
    'promo_lead_1', 'promo_lead_2',
    'promo_diff_1', 'promo_rolling_mean_7'
]
print(f'特徴量数: {len(features)} 列')


## 4. 33商品ファミリー別 3大GBDT学習

In [ ]:
families = train_df['family'].unique()
val_start_date = pd.to_datetime('2017-07-31')
train_start_date = pd.to_datetime('2016-01-01')
trn_mask = (all_df['date'] >= train_start_date) & (all_df['date'] < val_start_date)

trained_models = {}
for fam in families:
    fam_trn_mask = trn_mask & (all_df['family'] == fam)
    fam_train_df = all_df.loc[fam_trn_mask].dropna(subset=features)
    X_trn = fam_train_df[features].copy()
    y_trn = fam_train_df['log_sales'].values

    lgb_m = lgb.LGBMRegressor(objective='regression', metric='rmse', learning_rate=0.05, num_leaves=31, feature_fraction=0.8, random_state=42, verbose=-1, n_estimators=350)
    X_lgb = X_trn.copy()
    X_lgb['store_nbr'] = X_lgb['store_nbr'].astype('category')
    lgb_m.fit(X_lgb, y_trn)

    cb_m = cb.CatBoostRegressor(iterations=350, learning_rate=0.06, depth=6, loss_function='RMSE', task_type='CPU', random_seed=42, verbose=0)
    cb_m.fit(X_trn, y_trn, verbose=0)

    xgb_m = xgb.XGBRegressor(n_estimators=350, learning_rate=0.05, max_depth=5, subsample=0.8, colsample_bytree=0.8, tree_method='hist', random_state=42)
    xgb_m.fit(X_trn, y_trn, verbose=False)

    trained_models[fam] = {'lgb': lgb_m, 'cb': cb_m, 'xgb': xgb_m}
print('全33ファミリーのモデル学習完了！')


## 5. 再帰的推論ループ (Recursive Roll-forward Engine)

In [ ]:
def run_recursive_inference(df_source, start_date, end_date, is_val=True):
    pivot_sales = df_source.pivot(index='date', columns=['store_nbr', 'family'], values='log_sales').copy()
    dates = pd.date_range(start_date, end_date)
    step_predictions = []
    active_zero = val_zero_pairs if is_val else zero_pairs

    for current_date in dates:
        day_mask = (df_source['date'] == current_date)
        day_df = df_source.loc[day_mask].copy()

        for lag in [1, 2, 3, 7, 14, 21, 28, 364]:
            target_d = current_date - timedelta(days=lag)
            alt_d = current_date - timedelta(days=28)
            lag_series = pivot_sales.loc[target_d] if target_d in pivot_sales.index else pivot_sales.loc[alt_d]
            day_df[f'sales_lag_{lag}'] = day_df.set_index(['store_nbr', 'family']).index.map(lag_series).values

        d_1 = current_date - timedelta(days=1)
        for window in [7, 14, 28]:
            d_win = current_date - timedelta(days=window)
            mean_series = pivot_sales.loc[d_win:d_1].mean(axis=0)
            day_df[f'sales_rolling_mean_{window}'] = day_df.set_index(['store_nbr', 'family']).index.map(mean_series).values

        day_preds = {}
        for fam in families:
            fam_mask = (day_df['family'] == fam)
            if not fam_mask.any(): continue
            X_day = day_df.loc[fam_mask, features].copy()
            X_lgb = X_day.copy()
            X_lgb['store_nbr'] = X_lgb['store_nbr'].astype('category')

            p_lgb = trained_models[fam]['lgb'].predict(X_lgb)
            p_cb = trained_models[fam]['cb'].predict(X_day)
            p_xgb = trained_models[fam]['xgb'].predict(X_day)
            p_sales = np.clip(np.expm1(0.50 * p_lgb + 0.30 * p_cb + 0.20 * p_xgb), 0, None)

            for idx, (st, val) in enumerate(zip(day_df.loc[fam_mask, 'store_nbr'], p_sales)):
                if (st, fam) in active_zero: p_sales[idx] = 0.0
            for idx_orig, p_val in zip(day_df.loc[fam_mask].index, p_sales):
                day_preds[idx_orig] = p_val

        day_df['pred_sales'] = pd.Series(day_preds)
        step_predictions.append(day_df[['id', 'date', 'store_nbr', 'family', 'sales', 'pred_sales']])

        for _, row in day_df.iterrows():
            pivot_sales.loc[current_date, (row['store_nbr'], row['family'])] = np.log1p(row['pred_sales'])

    return pd.concat(step_predictions, ignore_index=True)

# 手元検証
val_res = run_recursive_inference(all_df, '2017-07-31', '2017-08-15', is_val=True)
val_rmsle = np.sqrt(mean_squared_error(np.log1p(val_res['sales']), np.log1p(val_res['pred_sales'])))
print(f'手元検証 Val RMSLE: {val_rmsle:.5f}')


## 6. テスト推論 & 提出ファイル作成

In [ ]:
test_res = run_recursive_inference(all_df, '2017-08-16', '2017-08-31', is_val=False)
sub = test_res[['id', 'pred_sales']].rename(columns={'pred_sales': 'sales'}).sort_values('id').reset_index(drop=True)
sub.to_csv('003_store-sales/submission_exp008.csv', index=False)
print('提出完了ファイル作成完了！')
print(sub.head(10))
